# 10. Consolidación de la corrida oficial de M2 y M3

Este cuaderno **no vuelve a entrenar**. Verifica por SHA-256 los pesos que respaldan los resultados documentados, los consolida en rutas activas portables y reconstruye la evidencia de validación. La corrida posterior con pesos diferentes se conserva en `historico/reentrenamiento_no_canonico_2026-10-25`.

## A. Localizar y comprobar la evidencia

- **Qué hace:** localiza los pesos, las métricas y la partición de la corrida oficial.
- **Para qué:** impedir que una ejecución distinta se presente como la utilizada en la tesis.
- **Por qué:** un mismo entrenamiento neuronal puede producir pesos y resultados diferentes si el entorno no es completamente determinista.

In [ ]:
%pip install -q pandas openpyxl matplotlib

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB')
else:
    RAIZ = Path.cwd()

HISTORICO = RAIZ / 'historico/restauracion_pre_clasificacion_2026-10-25'
MODELOS_H = HISTORICO / 'modelos'
TABLAS_H = HISTORICO / 'tablas/10_modelo_final_humano'
SALIDA = RAIZ / 'TABLAS/10_entrenamiento_boliviano'
M2_ACTIVO = RAIZ / 'outputs/modelos/robertuito_boliviano_m2_final'
M3_ACTIVO = RAIZ / 'outputs/modelos/robertuito_smps_boliviano_m3_final'
SALIDA.mkdir(parents=True, exist_ok=True)

FUENTES = {
    'M2_ajustado_boliviano': MODELOS_H / 'robertuito_boliviano_m2_v5/modelo_final',
    'M3_SMPS_boliviano': MODELOS_H / 'robertuito_smps_boliviano_m3_v5/modelo_final',
}
DESTINOS = {
    'M2_ajustado_boliviano': M2_ACTIVO,
    'M3_SMPS_boliviano': M3_ACTIVO,
}
HASHES_OFICIALES = {
    'M2_ajustado_boliviano': '250cc56b1577383bf769ae3b4e2e882cfb1aeaea71f5101917075ac0731db40b',
    'M3_SMPS_boliviano': '9ae315cbb3879a6baacd1770022cd6ddb5bac6d3e0ba35c7d8aeaec7f7ced8ba',
}

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()

for nombre, carpeta in FUENTES.items():
    peso = carpeta / 'model.safetensors'
    if not peso.is_file():
        raise FileNotFoundError(f'Falta el peso oficial de {nombre}.')
    if sha256(peso) != HASHES_OFICIALES[nombre]:
        raise ValueError(f'El hash histórico de {nombre} no coincide con el oficial.')

print('Fuentes oficiales verificadas por SHA-256.')
display(pd.DataFrame([
    {'modelo': nombre, 'sha256': HASHES_OFICIALES[nombre]}
    for nombre in HASHES_OFICIALES
]).head(10))

## B. Consolidar los pesos sin reentrenar

- **Qué hace:** copia los archivos oficiales cuando la ruta activa no existe y comprueba el peso resultante.
- **Para qué:** mantener rutas simples para la evaluación y el prototipo.
- **Por qué:** copiar pesos ya identificados no cambia el modelo ni genera una nueva corrida.

In [ ]:
ARCHIVOS_MODELO = [
    'config.json', 'model.safetensors', 'special_tokens_map.json',
    'tokenizer_config.json', 'tokenizer.json', 'training_args.bin',
]

for nombre, origen in FUENTES.items():
    destino = DESTINOS[nombre]
    destino.mkdir(parents=True, exist_ok=True)
    peso_destino = destino / 'model.safetensors'
    if peso_destino.is_file() and sha256(peso_destino) != HASHES_OFICIALES[nombre]:
        raise ValueError(
            f'La ruta activa de {nombre} contiene otro peso. Archívala antes de continuar.'
        )
    for archivo in ARCHIVOS_MODELO:
        fuente = origen / archivo
        if fuente.is_file() and not (destino / archivo).is_file():
            shutil.copy2(fuente, destino / archivo)
    if sha256(peso_destino) != HASHES_OFICIALES[nombre]:
        raise ValueError(f'La copia activa de {nombre} no conserva el hash oficial.')

print('Pesos oficiales disponibles en las rutas activas.')

## C. Recuperar validación, partición y receta

- **Qué hace:** lee la evidencia ejecutada y contrasta sus métricas con la tabla histórica.
- **Para qué:** reconstruir las salidas visibles sin recalcular el entrenamiento.
- **Por qué:** la evidencia guardada vincula cada F1 de validación con los pesos identificados.

In [ ]:
EVIDENCIAS = {
    'M2_ajustado_boliviano': MODELOS_H / 'robertuito_boliviano_m2_v5/evidencia_v5.json',
    'M3_SMPS_boliviano': MODELOS_H / 'robertuito_smps_boliviano_m3_v5/evidencia_v5.json',
}
METRICAS_H = TABLAS_H / 'metricas_validacion_v5.xlsx'
PARTICION_H = TABLAS_H / 'particion_boliviana_v5.xlsx'
metricas_h = pd.read_excel(METRICAS_H)
evidencias = {n: json.loads(p.read_text(encoding='utf-8')) for n, p in EVIDENCIAS.items()}

filas = []
for nombre, evidencia in evidencias.items():
    fila = {'modelo': nombre, **evidencia['metricas_validacion']}
    registrada = metricas_h.loc[metricas_h['modelo'] == nombre, 'eval_f1_macro'].iloc[0]
    if abs(float(fila['eval_f1_macro']) - float(registrada)) > 1e-12:
        raise ValueError(f'La evidencia y la tabla difieren para {nombre}.')
    filas.append(fila)
metricas = pd.DataFrame(filas)
metricas.to_excel(SALIDA / 'metricas_validacion.xlsx', index=False)
shutil.copy2(PARTICION_H, SALIDA / 'particion_boliviana.xlsx')

pesos = evidencias['M2_ajustado_boliviano']['pesos_clase']
receta = pd.DataFrame([
    {'elemento': 'Pesos de clase', 'M1_cuaderno_07': 'No', 'M2_M3_corrida_oficial': json.dumps(pesos, ensure_ascii=False)},
    {'elemento': 'Early stopping', 'M1_cuaderno_07': 'No', 'M2_M3_corrida_oficial': 'Paciencia 2'},
    {'elemento': 'Máximo de épocas', 'M1_cuaderno_07': 3, 'M2_M3_corrida_oficial': 3},
    {'elemento': 'Selección', 'M1_cuaderno_07': 'F1 macro de validación', 'M2_M3_corrida_oficial': 'F1 macro de validación'},
    {'elemento': 'Longitud máxima', 'M1_cuaderno_07': 128, 'M2_M3_corrida_oficial': 128},
])
receta.to_excel(SALIDA / 'diferencias_receta.xlsx', index=False)
display(metricas.head(10))
display(receta.head(10))

## D. Registrar selección, curvas y manifiesto

- **Qué hace:** selecciona por F1 macro de validación y guarda la procedencia verificable.
- **Para qué:** permitir que 11 y 14 carguen la misma corrida.
- **Por qué:** la selección no utiliza los 100 casos de prueba.

In [ ]:
seleccion = metricas.sort_values('eval_f1_macro', ascending=False).iloc[0]
seleccionado = str(seleccion['modelo'])
rutas = {
    nombre: str(destino.relative_to(RAIZ)).replace('\\', '/')
    for nombre, destino in DESTINOS.items()
}
seleccion_modelo = {
    'modelo_seleccionado': seleccionado,
    'ruta_modelo': rutas[seleccionado],
    'criterio': 'mayor F1 macro de validación de la corrida oficial',
    'f1_macro_validacion': float(seleccion['eval_f1_macro']),
    'rutas_evaluadas': rutas,
}
(SALIDA / 'seleccion_modelo.json').write_text(
    json.dumps(seleccion_modelo, ensure_ascii=False, indent=2), encoding='utf-8'
)

manifiesto = {
    'estado': 'corrida_oficial_consolidada_sin_reentrenamiento',
    'identificacion': 'pesos y métricas que respaldan la tesis v47/v48',
    'modelos': {
        nombre: {
            'ruta_activa': rutas[nombre],
            'sha256_model_safetensors': HASHES_OFICIALES[nombre],
            'f1_macro_validacion': float(metricas.loc[metricas.modelo == nombre, 'eval_f1_macro'].iloc[0]),
        }
        for nombre in HASHES_OFICIALES
    },
    'particion': {'entrenamiento': 1809, 'validacion': 454, 'semilla_particion': 280},
    'configuracion': {
        'semilla': 42, 'max_tokens': 128, 'epocas_maximas': 3,
        'tasa_aprendizaje': 2e-5, 'lote_entrenamiento': 16,
        'pesos_clase': pesos, 'early_stopping_paciencia': 2,
    },
    'reentrenamiento_posterior': 'historico/reentrenamiento_no_canonico_2026-10-25',
}
(SALIDA / 'manifest_corrida_oficial.json').write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8'
)

fig, ejes = plt.subplots(2, 2, figsize=(11, 8))
for columna, (nombre, evidencia) in enumerate(evidencias.items()):
    historial = pd.DataFrame(evidencia['historial'])
    ent = historial.dropna(subset=['loss']) if 'loss' in historial else pd.DataFrame()
    val = historial.dropna(subset=['eval_f1_macro']) if 'eval_f1_macro' in historial else pd.DataFrame()
    ejes[0, columna].plot(ent['epoch'], ent['loss'], marker='o')
    ejes[0, columna].set_title(f'Pérdida: {nombre}')
    ejes[0, columna].set_xlabel('Época')
    ejes[0, columna].set_ylabel('Pérdida')
    ejes[1, columna].plot(val['epoch'], val['eval_f1_macro'], marker='o', color='#2E8B57')
    ejes[1, columna].set_title(f'F1 macro: {nombre}')
    ejes[1, columna].set_xlabel('Época')
    ejes[1, columna].set_ylabel('F1 macro')
    ejes[1, columna].set_ylim(0, 1)
fig.tight_layout()
fig.savefig(SALIDA / 'curvas_entrenamiento.png', dpi=170, bbox_inches='tight')
plt.show()

print('Modelo seleccionado:', seleccionado)
print('No se ejecutó entrenamiento en este cuaderno.')
display(metricas[['modelo', 'eval_accuracy', 'eval_f1_macro']].head(10))